# Lesson 4.3 — The Claude Agent SDK, Hosting Models, and Hooks

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Describe what the Claude Agent SDK provides vs a custom loop
- Compare self-hosted and Anthropic-hosted agent deployment
- Explain hooks for deterministic actions and guardrails
- Sketch where hooks fire in an agent lifecycle

**You will produce:** A hook-registry demo that enforces deterministic pre/post actions around tool calls.

## Concepts

The **Claude Agent SDK** provides a batteries-included harness - the agent loop, tool wiring, context management, and extension points - so you do not rebuild everything from Lesson 4.2.

- **Deployment models**: **self-hosted** (you run the loop and infra, maximum control) vs **Anthropic-hosted** managed options (less ops burden). Choose based on control, compliance, and operational capacity.
- **Hooks**: deterministic callbacks that fire at defined lifecycle points (e.g., before/after a tool call). Use them to enforce guardrails, logging, approvals, or to block destructive actions - deterministically, not by asking the model nicely. Hooks are also a Domain 7 safety control.

> The Agent SDK API surface evolves; confirm exact classes, hook names, and hosting options at https://docs.anthropic.com before building.

> **Reference:** [Anthropic - Agent SDK & hooks](https://docs.anthropic.com/en/docs/agents-and-tools/agent-sdk/overview)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {prompt[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — A simple hook registry

Hooks run deterministically around tool execution.

In [ ]:
class HookRegistry:
    def __init__(self):
        self.pre, self.post = [], []
    def on_pre_tool(self, fn): self.pre.append(fn); return fn
    def on_post_tool(self, fn): self.post.append(fn); return fn
    def run_tool(self, name, inp, execute):
        for h in self.pre:
            decision = h(name, inp)
            if decision == "block":
                return {"blocked": True, "tool": name}
        result = execute(name, inp)
        for h in self.post:
            h(name, inp, result)
        return result

hooks = HookRegistry()
print("registry ready")

## Code-Along: Step 2 — Register a guardrail hook

Deterministically block a destructive tool regardless of model intent.

In [ ]:
@hooks.on_pre_tool
def block_destructive(name, inp):
    if name in {"delete_database", "wipe_disk"}:
        return "block"

@hooks.on_post_tool
def audit(name, inp, result):
    print(f"AUDIT: {name}({inp}) -> {result}")

def execute(name, inp):
    return f"ran {name}"

print(hooks.run_tool("delete_database", {"db": "prod"}, execute))
print(hooks.run_tool("calculator", {"expression": "1+1"}, execute))

## Exercise

Add a pre-hook that requires a confirmation token for any tool whose name starts with `send_` (e.g., `send_email`). Block execution if `inp.get('confirmed')` is not True.

In [ ]:
@hooks.on_pre_tool
def require_confirmation(name, inp):
    # TODO: if name.startswith('send_') and not inp.get('confirmed'): return 'block'
    pass

# TODO: test hooks.run_tool('send_email', {...}, execute) with and without confirmed=True

## Key Takeaways

- The Agent SDK provides a ready-made harness so you don't rebuild the loop.
- Self-hosted gives control; Anthropic-hosted reduces ops burden.
- Hooks enforce deterministic actions (guardrails, logging, approvals) around tool calls.
- Use hooks - not polite prompts - to block destructive actions.

---

**Next:** [Lesson 4.4 — Agent Patterns and Frameworks (Strands, LangGraph, PydanticAI)](./lesson-4.4-agent-patterns-frameworks.ipynb)